# 01 - Analise Exploratoria de Dados (EDA)

Pipeline completo: Carga do banco -> Validacao -> Filtro temporal -> Feature engineering -> Selecao de features

**Saidas:**
- `output/data/raw_sensor_data_YYYYMMDD_HHMMSS.csv` (backup dos dados brutos)
- `output/data/features_extracted_YYYYMMDD_HHMMSS.csv` (dataset de features)
- `output/data/features_latest.csv` (copia latest para notebook 02)
- `output/metrics/analise_exploratoria_summary.json`
- `config/feature_config.json` (features selecionadas + parametros)
- Figuras interativas em `output/figures/`

In [ ]:
# =============================================================================
# Celula 0: Fonte de dados (AUTO/SQL/CSV) + contexto de exportacao
# =============================================================================
import os
import sys
import json
from pathlib import Path

# --- Bootstrap para importar modulos compartilhados (notebooks/shared) ---
def _ensure_shared_path():
    cwd = Path.cwd()
    if (cwd / 'shared').exists():
        sys.path.insert(0, str(cwd))
    elif (cwd / 'notebooks' / 'shared').exists():
        sys.path.insert(0, str(cwd / 'notebooks'))


def _load_json_if_exists(path):
    if not os.path.exists(path):
        return {}
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}


_ensure_shared_path()

from shared.data_sources import select_raw_source

DATA_DIR = os.path.join('output', 'data')
METRICS_DIR = os.path.join('output', 'metrics')
os.makedirs(METRICS_DIR, exist_ok=True)

CONFIG_PATH = os.path.join(METRICS_DIR, 'data_source_config.json')
MANIFEST_PATH = os.path.join(METRICS_DIR, 'eda_input_manifest.json')

DATA_SOURCE, CSV_FILE = select_raw_source(
    data_dir=DATA_DIR,
    config_path=CONFIG_PATH,
    allow_sql=True,
    default_source='AUTO',
)

DATA_SOURCE_CONFIG = _load_json_if_exists(CONFIG_PATH)
EDA_INPUT_MANIFEST = _load_json_if_exists(MANIFEST_PATH)

REQUESTED_SOURCE = str(DATA_SOURCE_CONFIG.get('data_source', 'AUTO') or 'AUTO').upper()
RESOLVED_SOURCE = str(DATA_SOURCE or DATA_SOURCE_CONFIG.get('resolved_source') or 'SQL').upper()

manifest_csv = ''
if EDA_INPUT_MANIFEST:
    manifest_csv = os.path.basename(
        str(EDA_INPUT_MANIFEST.get('filename') or EDA_INPUT_MANIFEST.get('csv_path') or '')
    ).strip()

SOURCE_CONTEXT = {
    'requested_source': REQUESTED_SOURCE,
    'resolved_source': RESOLVED_SOURCE,
    'csv_file': CSV_FILE,
    'manifest_csv': manifest_csv or None,
    'manifest_export_id': EDA_INPUT_MANIFEST.get('export_id'),
    'manifest_collection_id': EDA_INPUT_MANIFEST.get('collection_id'),
    'manifest_device_id': EDA_INPUT_MANIFEST.get('device_id'),
    'manifest_rows': EDA_INPUT_MANIFEST.get('rows'),
}

print('=== CONTEXTO DA FONTE ===')
print(f"Fonte solicitada: {SOURCE_CONTEXT['requested_source']}")
print(f"Fonte resolvida:  {SOURCE_CONTEXT['resolved_source']}")
if SOURCE_CONTEXT['resolved_source'] == 'CSV':
    print(f"CSV ativo:        {SOURCE_CONTEXT.get('csv_file') or '--'}")
else:
    print('CSV ativo:        -- (consulta SQL no Oracle)')

if SOURCE_CONTEXT['manifest_export_id']:
    print('\n=== MANIFESTO DE EXPORTACAO (control.html) ===')
    print(f"Export ID:        {SOURCE_CONTEXT['manifest_export_id']}")
    print(f"Collection:       {SOURCE_CONTEXT.get('manifest_collection_id') or '--'}")
    print(f"Device:           {SOURCE_CONTEXT.get('manifest_device_id') or '--'}")
    print(f"Arquivo CSV:      {SOURCE_CONTEXT.get('manifest_csv') or '--'}")
    print(f"Rows manifest:    {SOURCE_CONTEXT.get('manifest_rows') or '--'}")
else:
    print('\nManifesto de exportacao ainda nao encontrado em output/metrics/eda_input_manifest.json')


In [ ]:
# =============================================================================
# Celula 1: Configuracao e Imports
# =============================================================================
import os
import sys
import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats as scipy_stats
from scipy.signal import savgol_filter
from scipy.stats import skew, kurtosis, kruskal
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import oracledb

warnings.filterwarnings('ignore')

# --- Bootstrap para importar modulos compartilhados (notebooks/shared) ---
def _ensure_shared_path():
    cwd = Path.cwd()
    if (cwd / 'shared').exists():
        sys.path.insert(0, str(cwd))
    elif (cwd / 'notebooks' / 'shared').exists():
        sys.path.insert(0, str(cwd / 'notebooks'))


def _load_json_if_exists(path):
    if not os.path.exists(path):
        return {}
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}


_ensure_shared_path()

from shared.class_config import (
    CLASS_ORDER, COLOR_MAP, LEGACY_CLASS_ORDER, LEGACY_COLOR_MAP,
    derive_composite_label, get_active_classes, get_color_map,
    FILTER_LABELS_SQL,
)
from shared.data_sources import load_raw_from_oracle

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# --- CONFIGURACAO DO BANCO (Oracle XE) ---
ORACLE_USER     = os.environ.get('ORACLE_USER', 'dersao')
ORACLE_PASSWORD = os.environ.get('ORACLE_PASSWORD', '986960440')
ORACLE_HOST     = os.environ.get('ORACLE_HOST', 'localhost')
ORACLE_PORT     = os.environ.get('ORACLE_PORT', '1521')
ORACLE_SERVICE  = os.environ.get('ORACLE_SERVICE_NAME', 'xepdb1')

# DSN para oracledb direto (thin mode): host:port/service funciona aqui
ORACLE_DSN_STR  = f'{ORACLE_HOST}:{ORACLE_PORT}/{ORACLE_SERVICE}'

# IMPORTANTE: SQLAlchemy trata "host:port/nome" como SID (legado).
# Para SERVICE_NAME obrigatoriamente usar "?service_name=..."
ORACLE_CONN_STR = (
    f'oracle+oracledb://{ORACLE_USER}:{ORACLE_PASSWORD}'
    f'@{ORACLE_HOST}:{ORACLE_PORT}/?service_name={ORACLE_SERVICE}'
)

# =============================================================================
# Contexto de controle + manifesto de exportacao
# =============================================================================
CONTROL_STATE_PATH = os.path.join(
    '..', 'api', 'control_states', 'control_state_ESP32_MPU6050_ORACLE.json'
)
if not os.path.exists(CONTROL_STATE_PATH):
    CONTROL_STATE_PATH = os.path.join('..', 'api', 'control_state.json')

if 'MANIFEST_PATH' not in globals():
    MANIFEST_PATH = os.path.join('output', 'metrics', 'eda_input_manifest.json')

control_state = _load_json_if_exists(CONTROL_STATE_PATH)
EDA_INPUT_MANIFEST = _load_json_if_exists(MANIFEST_PATH)

if control_state:
    print(f'[AUTO] control_state carregado: {CONTROL_STATE_PATH}')
    print(f"  Modo atual:      {control_state.get('mode', 'N/A')}")
    print(f"  Sample rate:     {control_state.get('sample_rate', 'N/A')} Hz")
    print(f"  Collection ID:   {control_state.get('collection_id', 'N/A')}")
else:
    print(f'[AVISO] control_state nao encontrado em {CONTROL_STATE_PATH}')

SAMPLE_RATE_HZ = int(control_state.get('sample_rate', 20)) if control_state else 20
EFFECTIVE_SAMPLE_RATE_HZ = SAMPLE_RATE_HZ

# --- PARAMETROS DE FILTRO (opcionais) ---
FILTER_SAMPLE_RATE_HZ = None  # ex: 20 para filtrar; None = usar todas
FILTER_COLLECTION_ID  = None  # ex: 'col_...' ou ['col_a', 'col_b']
SQL_DEVICE_ID_FILTER  = None  # opcional para filtrar por device no SQL

# --- TABELA DE DADOS (arquitetura de duas tabelas) ---
DATA_TABLE          = 'sensor_training_data'   # coletas supervisionadas (padrao)
DATA_TABLE_FALLBACK = 'sensor_data'            # legado (fallback se training vazia)

# 7 classes compostas (derivadas de cmd_speed_label + rot_state_label)
FILTER_LABELS = CLASS_ORDER

_manifest_collection = str(EDA_INPUT_MANIFEST.get('collection_id') or '').strip()
_manifest_device = str(EDA_INPUT_MANIFEST.get('device_id') or '').strip()
_manifest_rate_min = EDA_INPUT_MANIFEST.get('sample_rate_min')
_manifest_rate_max = EDA_INPUT_MANIFEST.get('sample_rate_max')

if DATA_SOURCE == 'SQL':
    if FILTER_COLLECTION_ID is None and _manifest_collection:
        FILTER_COLLECTION_ID = _manifest_collection
    if _manifest_device:
        SQL_DEVICE_ID_FILTER = _manifest_device
else:
    if (
        _manifest_rate_min is not None and _manifest_rate_max is not None and
        float(_manifest_rate_min) == float(_manifest_rate_max)
    ):
        EFFECTIVE_SAMPLE_RATE_HZ = float(_manifest_rate_min)


def _rows_to_dict(cursor, rows):
    cols = [d[0].lower() for d in cursor.description]
    return [dict(zip(cols, row)) for row in rows]


def _sql_filters(base_states=('LOW', 'MEDIUM', 'HIGH', 'OFF')):
    where = [
        "NVL(cmd_speed_label, fan_state) IN ({})".format(
            ', '.join([f"'{s}'" for s in base_states])
        )
    ]
    params = {}
    if isinstance(FILTER_COLLECTION_ID, str) and FILTER_COLLECTION_ID.strip():
        where.append('collection_id = :collection_id')
        params['collection_id'] = FILTER_COLLECTION_ID.strip()
    if SQL_DEVICE_ID_FILTER:
        where.append('device_id = :device_id')
        params['device_id'] = SQL_DEVICE_ID_FILTER
    return ' AND '.join(where), params


if DATA_SOURCE == 'SQL':
    print('\n[AUTO] Fonte SQL selecionada: consultando Oracle para validar parametros...')
    try:
        _conn_auto = oracledb.connect(
            user=ORACLE_USER,
            password=ORACLE_PASSWORD,
            dsn=ORACLE_DSN_STR,
        )
        _cursor = _conn_auto.cursor()

        # Detectar tabela com dados (training primeiro, fallback legado)
        _eda_table = DATA_TABLE
        try:
            _cursor.execute(f'SELECT COUNT(*) FROM {_eda_table} WHERE ROWNUM <= 1')
            _cnt_tr = _cursor.fetchone()[0]
            if _cnt_tr == 0:
                _cursor.execute(f'SELECT COUNT(*) FROM {DATA_TABLE_FALLBACK} WHERE ROWNUM <= 1')
                _cnt_fb = _cursor.fetchone()[0]
                if _cnt_fb > 0:
                    _eda_table = DATA_TABLE_FALLBACK
                    print(f'[INFO] {DATA_TABLE} vazia; usando fallback {DATA_TABLE_FALLBACK}')
        except Exception as _te:
            _eda_table = DATA_TABLE_FALLBACK
            print(f'[AVISO] Nao foi possivel verificar tabelas: {_te}')
        print(f'[INFO] Tabela Oracle (EDA stats): {_eda_table}')

        _where_main, _params_main = _sql_filters(('LOW', 'MEDIUM', 'HIGH', 'OFF'))
        _cursor.execute(
            f"""
            SELECT NVL(cmd_speed_label, fan_state) AS fan_state,
                   COUNT(*) AS cnt,
                   MIN(ts_epoch) AS ts_min,
                   MAX(ts_epoch) AS ts_max,
                   AVG(sample_rate) AS avg_sample_rate
            FROM {_eda_table}
            WHERE {_where_main}
            GROUP BY NVL(cmd_speed_label, fan_state)
            ORDER BY cnt DESC
            """,
            _params_main,
        )
        _class_stats = _rows_to_dict(_cursor, _cursor.fetchall())

        _where_train, _params_train = _sql_filters(('LOW', 'MEDIUM', 'HIGH'))
        _cursor.execute(
            f"""
            SELECT collection_id, COUNT(*) AS cnt,
                   MIN(ts_epoch) AS ts_min, MAX(ts_epoch) AS ts_max
            FROM {_eda_table}
            WHERE {_where_train}
            GROUP BY collection_id
            ORDER BY MAX(ts_epoch) DESC
            """,
            _params_train,
        )
        _collection_stats = _rows_to_dict(_cursor, _cursor.fetchall())

        _cursor.execute(
            f"""
            SELECT ROUND(AVG(sample_rate), 2) AS db_avg_rate,
                   ROUND(MIN(sample_rate), 2) AS db_min_rate,
                   ROUND(MAX(sample_rate), 2) AS db_max_rate
            FROM {_eda_table}
            WHERE {_where_train}
            """,
            _params_train,
        )
        _rate_rows = _rows_to_dict(_cursor, _cursor.fetchall())
        _rate_stats = _rate_rows[0] if _rate_rows else {}

        _conn_auto.close()

        print(f'\n{"="*60}')
        print(' PARAMETROS AUTO-DETECTADOS DO SISTEMA (Oracle)')
        print(f'{"="*60}')
        print(f'  SAMPLE_RATE_HZ (control_state): {SAMPLE_RATE_HZ} Hz')
        if _rate_stats and _rate_stats.get('db_avg_rate') is not None:
            print(f"  sample_rate no banco (media):   {_rate_stats['db_avg_rate']} Hz")
            print(f"  sample_rate no banco (min/max): {_rate_stats['db_min_rate']} / {_rate_stats['db_max_rate']} Hz")
            db_rate = float(_rate_stats['db_avg_rate'])
            if abs(db_rate - SAMPLE_RATE_HZ) / max(SAMPLE_RATE_HZ, 1) > 0.15:
                print('  *** ALERTA: Taxa no banco difere do control_state. ***')

        print('\n  Dados por cmd_speed_label no banco:')
        _total_training = 0
        for row in _class_stats:
            state = row['fan_state']
            count = int(row['cnt'])
            duration = (float(row['ts_max']) - float(row['ts_min'])) / 60 if row['ts_max'] and row['ts_min'] else 0
            marker = ' (inativo)' if state == 'OFF' else ''
            print(f'    {state:8s}: {count:6d} amostras, {duration:6.1f} min{marker}')
            if state in ('LOW', 'MEDIUM', 'HIGH'):
                _total_training += count
        print(f'    {"TOTAL":8s}: {_total_training:6d} amostras de treino')

        if _collection_stats:
            print('\n  Collections (treino):')
            for row in _collection_stats:
                duration = (float(row['ts_max']) - float(row['ts_min'])) / 60 if row['ts_max'] and row['ts_min'] else 0
                print(f"    {row['collection_id']}: {int(row['cnt'])} amostras, {duration:.1f} min")

        print(f'{"="*60}')

    except Exception as e:
        print(f'[AVISO] Nao foi possivel consultar Oracle: {e}')
        print(f'  Usando SAMPLE_RATE_HZ = {SAMPLE_RATE_HZ} do control_state.')
else:
    print('\n[AUTO] Fonte CSV selecionada: auto-detect Oracle foi ignorado nesta execucao.')

# --- PARAMETROS DE FEATURE ENGINEERING ---
WINDOW_SIZE = 100    # pontos por janela
STEP_SIZE = 20       # stride da janela deslizante

# Feature selection
ANOVA_ALPHA = 0.05
CORRELATION_THRESHOLD = 0.85

# Eixos do sensor
SENSOR_AXES = ['accel_x_g', 'accel_y_g', 'accel_z_g', 'gyro_x_dps', 'gyro_y_dps', 'gyro_z_dps']

# Eixos/metricas candidatos do modelo
DERIVED_AXES = ['vibration_dps', 'accel_mag_g']
FEATURE_AXES = SENSOR_AXES + DERIVED_AXES
FEATURE_METRICS = ['std', 'range', 'rms']
FEATURE_COLUMNS = [f'{ax}_{m}' for ax in FEATURE_AXES for m in FEATURE_METRICS]

# Suavizacao para graficos (apenas visual)
SMOOTH_WINDOW_S = 0.5
SMOOTH_POLYORDER = 3

# Paths
from shared.paths import get_paths
PATHS = get_paths()
TIMESTAMP_STR = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DATA = str(PATHS.output_data)
OUTPUT_FIGURES = str(PATHS.output_figures)
OUTPUT_METRICS = str(PATHS.output_metrics)
CONFIG_DIR = str(PATHS.config_dir)


def _savgol_smooth(series, sample_rate_hz, window_s=SMOOTH_WINDOW_S, polyorder=SMOOTH_POLYORDER):
    values = np.asarray(series, dtype=float)
    if values.size < 3:
        return values
    window_len = int(round(window_s * sample_rate_hz))
    if window_len < 3:
        window_len = 3
    if window_len % 2 == 0:
        window_len += 1
    if window_len > values.size:
        window_len = values.size if values.size % 2 == 1 else values.size - 1
    if window_len < 3:
        return values
    if polyorder >= window_len:
        polyorder = max(1, window_len - 1)
    try:
        return savgol_filter(values, window_length=window_len, polyorder=polyorder)
    except Exception:
        return values

print('\n--- Configuracao Final ---')
print(f"Fonte dados:          {DATA_SOURCE}")
print(f'Timestamp:            {TIMESTAMP_STR}')
print(f'Oracle DSN:           {ORACLE_DSN_STR}')
print(f'Sample Rate (base):   {SAMPLE_RATE_HZ} Hz')
if FILTER_SAMPLE_RATE_HZ is not None:
    print(f'Filtro sample_rate:   {FILTER_SAMPLE_RATE_HZ} Hz')
if FILTER_COLLECTION_ID:
    print(f'Filtro collection_id: {FILTER_COLLECTION_ID}')
if SQL_DEVICE_ID_FILTER:
    print(f'Filtro device_id:     {SQL_DEVICE_ID_FILTER}')
if FILTER_LABELS:
    print(f'Filtro labels:        {FILTER_LABELS}')
print(f'Window: {WINDOW_SIZE} pontos = {WINDOW_SIZE/max(SAMPLE_RATE_HZ, 1):.0f}s, Step: {STEP_SIZE/max(SAMPLE_RATE_HZ, 1):.0f}s')
print(f'Figuras:  {os.path.abspath(OUTPUT_FIGURES)}')
print(f'Metricas: {os.path.abspath(OUTPUT_METRICS)}')

# --- IDs de rastreabilidade (usados na Secao C ? exportacao filtrada) ---
from datetime import datetime as _dt_eda
EDA_ID = f'eda_{_dt_eda.now().strftime("%Y%m%d_%H%M%S")}'
if isinstance(FILTER_COLLECTION_ID, str) and FILTER_COLLECTION_ID.strip():
    COLLECTION_ID = FILTER_COLLECTION_ID.strip()
elif _manifest_collection:
    COLLECTION_ID = _manifest_collection
else:
    COLLECTION_ID = control_state.get('collection_id', 'unknown') if control_state else 'unknown'
SAMPLE_HZ = float(EFFECTIVE_SAMPLE_RATE_HZ)   # alias para Secao C
print(f'EDA_ID:        {EDA_ID}')
print(f'COLLECTION_ID: {COLLECTION_ID}')


In [ ]:
# =============================================================================
# Celula 2: Carga do banco de dados
# =============================================================================
if 'DATA_SOURCE' not in globals():
    DATA_SOURCE = 'SQL'
if 'CSV_FILE' not in globals():
    CSV_FILE = None

RAW_SOURCE_PATH = None


def _apply_filters_df(df):
    """Aplica filtros no DataFrame. Deriva composite label antes de filtrar."""
    df_filtered = df.copy()
    df_filtered = derive_composite_label(df_filtered)

    if 'fan_state' in df_filtered.columns:
        if FILTER_LABELS:
            df_filtered = df_filtered[df_filtered['fan_state'].isin(FILTER_LABELS)].copy()
        else:
            df_filtered = df_filtered[df_filtered['fan_state'].isin(LEGACY_CLASS_ORDER)].copy()
    if FILTER_COLLECTION_ID and 'collection_id' in df_filtered.columns:
        if isinstance(FILTER_COLLECTION_ID, (list, tuple, set)):
            df_filtered = df_filtered[df_filtered['collection_id'].isin(list(FILTER_COLLECTION_ID))].copy()
        else:
            df_filtered = df_filtered[df_filtered['collection_id'] == FILTER_COLLECTION_ID].copy()
    if FILTER_SAMPLE_RATE_HZ is not None and 'sample_rate' in df_filtered.columns:
        df_filtered = df_filtered[df_filtered['sample_rate'] == FILTER_SAMPLE_RATE_HZ].copy()
    return df_filtered


if DATA_SOURCE == 'CSV':
    data_dir = OUTPUT_DATA if 'OUTPUT_DATA' in globals() else os.path.join('output', 'data')
    if not CSV_FILE:
        raise ValueError('Fonte CSV selecionada, mas nenhum arquivo foi escolhido. Use a Celula 0.')
    csv_path = os.path.join(data_dir, CSV_FILE)
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f'CSV nao encontrado: {csv_path}')
    df_raw = pd.read_csv(csv_path)
    RAW_SOURCE_PATH = csv_path
    df_raw = _apply_filters_df(df_raw)
    print(f'CSV carregado: {csv_path}')
else:
    # --- Carga via Oracle usando load_raw_from_oracle() de shared/data_sources.py ---
    # Passa collection_id como string para filtro SQL (filtros de lista/sample_rate
    # sao aplicados em Python via _apply_filters_df abaixo).
    _cid = FILTER_COLLECTION_ID if isinstance(FILTER_COLLECTION_ID, str) else None
    _device = SQL_DEVICE_ID_FILTER if isinstance(SQL_DEVICE_ID_FILTER, str) and SQL_DEVICE_ID_FILTER.strip() else None

    _oracle_table    = DATA_TABLE if 'DATA_TABLE' in dir() else 'sensor_training_data'
    _oracle_fallback = DATA_TABLE_FALLBACK if 'DATA_TABLE_FALLBACK' in dir() else 'sensor_data'
    df_raw = load_raw_from_oracle(
        connection_str=ORACLE_CONN_STR,
        device_id=_device,
        collection_id=_cid,
        table=_oracle_table,
    )
    # Fallback: se training vazia, tenta sensor_data legado
    if df_raw.empty and _oracle_table != _oracle_fallback:
        print(f'[INFO] {_oracle_table} sem dados; tentando fallback {_oracle_fallback}')
        df_raw = load_raw_from_oracle(
            connection_str=ORACLE_CONN_STR,
            device_id=_device,
            collection_id=_cid,
            table=_oracle_fallback,
        )
        _oracle_table = _oracle_fallback
    RAW_SOURCE_PATH = f'Oracle:{ORACLE_DSN_STR} (tabela: {_oracle_table})'

    # Aplicar filtros em Python (labels compostos, sample_rate, collection_id lista)
    df_raw = _apply_filters_df(df_raw)
    print(f'Dados carregados do Oracle: tabela={_oracle_table} (com labels compostos).')

if df_raw.empty:
    raise ValueError('Nenhum dado encontrado apos aplicar os filtros.')

# Normalizar tipos
if 'sample_rate' in df_raw.columns:
    df_raw['sample_rate'] = pd.to_numeric(df_raw['sample_rate'], errors='coerce')

# Atualizar taxa efetiva (usa filtro ou taxa unica no banco)
if 'sample_rate' in df_raw.columns and df_raw['sample_rate'].notna().any():
    unique_rates = sorted(df_raw['sample_rate'].dropna().unique().tolist())
    if FILTER_SAMPLE_RATE_HZ is not None:
        EFFECTIVE_SAMPLE_RATE_HZ = float(FILTER_SAMPLE_RATE_HZ)
    elif len(unique_rates) == 1:
        EFFECTIVE_SAMPLE_RATE_HZ = float(unique_rates[0])
    else:
        print(f'[AVISO] Multiplas sample_rate no dataset: {unique_rates}')
        print('[AVISO] Recomenda-se filtrar por sample_rate para consistencia.')
else:
    print('[AVISO] Coluna sample_rate nao encontrada; usando control_state.')

# --- Eixos derivados (para enriquecer pool de features) ---
if 'vibration_dps' not in df_raw.columns and 'vibration' in df_raw.columns:
    df_raw['vibration_dps'] = pd.to_numeric(df_raw['vibration'], errors='coerce')

if 'accel_mag_g' not in df_raw.columns and all(c in df_raw.columns for c in ('accel_x_g', 'accel_y_g', 'accel_z_g')):
    ax = pd.to_numeric(df_raw['accel_x_g'], errors='coerce')
    ay = pd.to_numeric(df_raw['accel_y_g'], errors='coerce')
    az = pd.to_numeric(df_raw['accel_z_g'], errors='coerce')
    df_raw['accel_mag_g'] = np.sqrt(ax**2 + ay**2 + az**2)

# Determinar classes ativas e cores baseado nos dados carregados
ACTIVE_CLASSES = get_active_classes(df_raw)
ACTIVE_COLOR_MAP = get_color_map(df_raw)

print(f'[INFO] Fonte ativa: {DATA_SOURCE}')
print(f'[INFO] Amostras carregadas: {len(df_raw)}')
print(f'[INFO] EFFECTIVE_SAMPLE_RATE_HZ: {EFFECTIVE_SAMPLE_RATE_HZ} Hz')
print(f'[INFO] Classes ativas: {ACTIVE_CLASSES}')
print(f'[INFO] Distribuicao: {df_raw["fan_state"].value_counts().to_dict()}')


In [ ]:
# =============================================================================
# Celula 3: Ajustes de tempo + politica de backup CSV
# =============================================================================
# Adicionar timestamp ISO 8601 legivel
_df_ts = pd.to_datetime(df_raw['timestamp'], unit='s', utc=True, errors='coerce')
if _df_ts.isna().all():
    df_raw['timestamp_iso'] = None
else:
    df_raw['timestamp_iso'] = _df_ts.dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')

# Adicionar informacao de frequencia de amostragem
df_raw['sample_rate_configured_hz'] = EFFECTIVE_SAMPLE_RATE_HZ

# Calcular taxa real de amostragem por classe
ts_col = pd.to_numeric(df_raw['timestamp'], errors='coerce').copy()
if ts_col.median() > 1e12:
    ts_col = ts_col / 1000.0

for cls in df_raw['fan_state'].dropna().unique():
    mask = df_raw['fan_state'] == cls
    t0 = ts_col[mask].min()
    duration = ts_col[mask].max() - t0
    n_samples = mask.sum()
    real_rate = n_samples / duration if duration > 0 else 0
    df_raw.loc[mask, 'sample_rate_real_hz'] = round(real_rate, 2)

# Politica de backup:
# - SQL: salva snapshot raw_sensor_data_<timestamp>.csv
# - CSV: evita duplicacao e reaproveita o proprio arquivo origem
if DATA_SOURCE == 'CSV' and RAW_SOURCE_PATH and os.path.exists(RAW_SOURCE_PATH):
    raw_csv_path = RAW_SOURCE_PATH
    print(f'[INFO] Fonte CSV: backup adicional nao criado. Usando arquivo existente: {raw_csv_path}')
else:
    raw_csv_path = os.path.join(OUTPUT_DATA, f'raw_sensor_data_{TIMESTAMP_STR}.csv')
    df_raw.to_csv(raw_csv_path, index=False)
    print(f'Dados brutos salvos em: {raw_csv_path}')

print(f'Shape: {df_raw.shape}')
print(f'Colunas: {list(df_raw.columns)}')
if len(df_raw) > 0 and 'timestamp_iso' in df_raw.columns:
    print(f'Exemplo timestamp_iso: {df_raw["timestamp_iso"].iloc[0]}')
print(f'Taxa configurada: {EFFECTIVE_SAMPLE_RATE_HZ} Hz')
print('Taxa real por classe:')
for cls in ACTIVE_CLASSES:
    if cls in df_raw['fan_state'].unique():
        rate = df_raw.loc[df_raw['fan_state'] == cls, 'sample_rate_real_hz'].iloc[0]
        print(f'  {cls}: {rate} Hz')
    else:
        print(f'  {cls}: n/a')


In [ ]:
# =============================================================================
# Celula 4: Validacao e estatisticas gerais
# =============================================================================
from IPython.display import display

print('=== INFO ===')
df_raw.info()
print()
print('=== NULLS ===')
print(df_raw.isnull().sum())
print()
print('=== DESCRIBE (GLOBAL) ===')
display(df_raw[SENSOR_AXES].describe())
print()
print('=== DESCRIBE POR CLASSE (fan_state) ===')
for cls in ACTIVE_CLASSES:
    subset = df_raw[df_raw['fan_state'] == cls]
    if subset.empty:
        print(f'--- {cls}: sem dados ---')
        continue
    print(f'--- {cls} (n={len(subset)}) ---')
    display(subset[SENSOR_AXES].describe())
    print()

In [ ]:
# =============================================================================
# Celula 5: Distribuicao de classes (barplot)
# =============================================================================
class_counts = df_raw['fan_state'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
order = ACTIVE_CLASSES
bar_colors = [ACTIVE_COLOR_MAP.get(c, '#888888') for c in order]
counts_ordered = [class_counts.get(c, 0) for c in order]

ax.bar(order, counts_ordered, color=bar_colors)
ax.set_title('Distribuicao das Amostras por Classe', fontsize=16)
ax.set_xlabel('Fan State')
ax.set_ylabel('Amostras')
for i, v in enumerate(counts_ordered):
    ax.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_FIGURES, '01_distribuicao_classes.png'), dpi=150)
plt.show()
print(class_counts)

In [ ]:
# =============================================================================
# Celula 6: Normalizacao do tempo relativo por classe + taxa real
# =============================================================================
df_raw['timestamp_s'] = df_raw['timestamp'].copy()
# Se timestamp estiver em ms, converter para s
if df_raw['timestamp_s'].median() > 1e12:
    df_raw['timestamp_s'] = df_raw['timestamp_s'] / 1000.0

# Tempo relativo por classe (segundos desde o inicio da coleta daquela classe)
df_raw['relative_time_s'] = 0.0
for cls in df_raw['fan_state'].unique():
    mask = df_raw['fan_state'] == cls
    t0 = df_raw.loc[mask, 'timestamp_s'].min()
    df_raw.loc[mask, 'relative_time_s'] = df_raw.loc[mask, 'timestamp_s'] - t0

# Duracao e taxa real por classe
duration_per_class = df_raw.groupby('fan_state')['relative_time_s'].max()
samples_per_class = df_raw.groupby('fan_state').size()

print('=== Duracao por classe (s) ===')
print(duration_per_class)
print()
print(f'Amostras por classe: {samples_per_class.to_dict()}')
print()
print('=== Taxa real de amostragem (Hz) ===')
for cls in ACTIVE_CLASSES:
    if cls not in samples_per_class or cls not in duration_per_class or duration_per_class[cls] == 0:
        print(f'  {cls}: n/a')
        continue
    real_rate = samples_per_class[cls] / duration_per_class[cls]
    print(f'  {cls}: {real_rate:.2f} Hz (configurada: {EFFECTIVE_SAMPLE_RATE_HZ} Hz)')

In [ ]:
# =============================================================================
# Celula 7: Calculo da janela de tempo comum
# =============================================================================
common_window_s = duration_per_class.min()
print(f'Janela de tempo comum: {common_window_s:.2f} s')
print(f'Isso corresponde a ~{common_window_s/60:.1f} minutos')

In [ ]:
# =============================================================================
# Celula 8: FILTRO TEMPORAL - recortar TODAS as classes pela janela comum
# ANTES de qualquer analise posterior (graficos E features)
# =============================================================================
df = df_raw[df_raw['relative_time_s'] <= common_window_s].copy()

print(f'Amostras ANTES do filtro: {len(df_raw)}')
print(f'Amostras APOS o filtro:  {len(df)}')
print(f'Amostras por classe (filtrado): {df.groupby("fan_state").size().to_dict()}')
print(f'Duracao por classe (filtrado):')
print(df.groupby('fan_state')['relative_time_s'].max())

In [ ]:
# =============================================================================
# Celula 8B: Separabilidade (Cohen's d) - sinais brutos (dados filtrados)
# =============================================================================
from IPython.display import display
from itertools import combinations
from shared.feature_selection import cohens_d

axes_for_d = []
axes_for_d += [ax for ax in (SENSOR_AXES or []) if ax in df.columns]
if 'DERIVED_AXES' in globals():
    axes_for_d += [ax for ax in (DERIVED_AXES or []) if ax in df.columns]

pairs = list(combinations(ACTIVE_CLASSES, 2))
rows = []
for ax in axes_for_d:
    row = {'signal': ax}
    d_values = []
    for a, b in pairs:
        g1 = df[df['fan_state'] == a][ax]
        g2 = df[df['fan_state'] == b][ax]
        d_val = cohens_d(g1, g2)
        row[f'd_{a}_{b}'] = d_val
        d_values.append(d_val)
    row['d_min_all'] = float(min(d_values)) if d_values else 0.0
    rows.append(row)

df_d = pd.DataFrame(rows).sort_values('d_min_all', ascending=False)

print(f"COHEN'S D ENTRE CLASSES (SINAIS BRUTOS, DADOS FILTRADOS) - {len(pairs)} pares")
print('=' * 80)
print('Interpretacao: |d| > 0.8 = grande, |d| > 0.5 = medio, |d| > 0.2 = pequeno')
print('=' * 80)

# Mostrar colunas resumidas (signal + d_min_all + pares individuais se poucos)
show_cols = ['signal', 'd_min_all']
if len(pairs) <= 6:
    show_cols = ['signal'] + [f'd_{a}_{b}' for a, b in pairs] + ['d_min_all']
display(df_d[show_cols].round(3))

if not df_d.empty:
    n_large = int((df_d['d_min_all'] >= 0.8).sum())
    n_medium = int((df_d['d_min_all'] >= 0.5).sum())
    print(f"\nResumo (d_min_all): >=0.8={n_large}  >=0.5={n_medium}  total={len(df_d)}")

In [ ]:
# =============================================================================
# Celula 9: Graficos interativos individuais (Plotly) - dados filtrados
# =============================================================================
print('Gerando graficos individuais por eixo...')

# Paleta consistente com o grafico espectral (Cell 14)
TEMPORAL_COLORS = {
    'FAN_OFF':        '#7f7f7f',
    'LOW_ROT_ON':     '#1f77b4',
    'LOW_ROT_OFF':    '#ff7f0e',
    'MEDIUM_ROT_ON':  '#2ca02c',
    'MEDIUM_ROT_OFF': '#9467bd',
    'HIGH_ROT_ON':    '#d62728',
    'HIGH_ROT_OFF':   '#17becf',
}

classes = list(reversed(ACTIVE_CLASSES))  # HIGH primeiro no topo
colors = TEMPORAL_COLORS
SLIDER_BORDERWIDTH = 1  # Plotly requires int for rangeslider borderwidth
slider_style = dict(
    visible=True,
    thickness=0.01,
    bgcolor='rgba(0,0,0,0.03)',
    bordercolor='rgba(0,0,0,0.2)',
    borderwidth=int(SLIDER_BORDERWIDTH),
)
window_options = [(0.5, '.5s'), (1, '1s'), (2, '2s'), (5, '5s'), (10, '10s')]

for axis in SENSOR_AXES:
    # Filtrar classes com dados para este eixo
    classes_with_data = [c for c in classes if not df[df['fan_state'] == c].empty]
    if not classes_with_data:
        continue

    fig = make_subplots(
        rows=len(classes_with_data), cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=classes_with_data,
    )

    for i, cls in enumerate(classes_with_data, start=1):
        subset = df[df['fan_state'] == cls]
        y_smooth = _savgol_smooth(
            subset[axis].values,
            sample_rate_hz=EFFECTIVE_SAMPLE_RATE_HZ,
            window_s=SMOOTH_WINDOW_S,
            polyorder=SMOOTH_POLYORDER,
        )
        is_off = cls.endswith('ROT_OFF')
        fig.add_trace(
            go.Scatter(
                x=subset['relative_time_s'],
                y=y_smooth,
                mode='lines',
                name=cls,
                line=dict(
                    color=colors.get(cls, '#888'),
                    width=2.2 if is_off else 2.0,
                    dash='dash' if is_off else 'solid',
                ),
                hovertemplate=f'Classe={cls}<br>Tempo (s)=%{{x:.3f}}<br>Valor=%{{y:.6f}}<extra></extra>',
            ),
            row=i, col=1,
        )
        fig.update_yaxes(title_text=axis, row=i, col=1)

    fig.update_xaxes(title_text='Tempo (s)', row=len(classes_with_data), col=1)
    fig.update_xaxes(rangeslider=slider_style, row=len(classes_with_data), col=1)

    x_min = df['relative_time_s'].min()
    x_max = df['relative_time_s'].max()
    buttons = []
    for window_s, label in window_options:
        start = max(x_max - window_s, x_min)
        xaxis_args = {f'xaxis{k}.range' if k > 1 else 'xaxis.range': [start, x_max]
                      for k in range(1, len(classes_with_data) + 1)}
        buttons.append(
            dict(label=label, method='relayout', args=[xaxis_args])
        )

    fig.update_layout(
        title_text=f'Assinatura Individual (Suavizada): {axis}',
        height=max(300 * len(classes_with_data), 600),
        showlegend=False,
        margin=dict(t=90),
        updatemenus=[
            dict(
                type='buttons',
                direction='right',
                x=0, y=1.15,
                xanchor='left', yanchor='top',
                showactive=True,
                buttons=buttons,
                pad={'r': 10, 't': 10},
            )
        ],
    )
    fig.write_html(os.path.join(OUTPUT_FIGURES, f'02_interactive_individual_{axis}.html'))
    fig.show()
    print(f'  Salvo: 02_interactive_individual_{axis}.html')

In [ ]:
# =============================================================================
# Celula 10: Graficos interativos sobrepostos (Plotly) — estilo espectral
# =============================================================================
print('Gerando graficos sobrepostos por eixo...')

# Paleta identica ao grafico espectral (TEMPORAL_COLORS definido na celula anterior)
SHORT_TEMPORAL_LABELS = {
    'FAN_OFF': 'FAN OFF', 'LOW_ROT_ON': 'LO ON', 'LOW_ROT_OFF': 'LO OFF',
    'MEDIUM_ROT_ON': 'MD ON', 'MEDIUM_ROT_OFF': 'MD OFF',
    'HIGH_ROT_ON': 'HI ON', 'HIGH_ROT_OFF': 'HI OFF',
}

def temp_line_style(cls):
    is_off = cls.endswith('ROT_OFF')
    return dict(
        color=TEMPORAL_COLORS.get(cls, '#888'),
        width=2.2 if is_off else 2.0,
        dash='dash' if is_off else 'solid',
    )

# Layout — mesmo esquema de coordenadas do grafico espectral
_H_T, _MT_T, _MB_T = 720, 310, 185
_plot_bot_T = _MB_T / _H_T
_plot_h_T   = 1 - _MT_T / _H_T - _plot_bot_T
_title_y_T  = round(_plot_bot_T + 1.08 * _plot_h_T, 4)

SLIDER_BORDERWIDTH = 1
slider_style = dict(
    visible=True,
    thickness=0.03,
    bgcolor='rgba(0,0,0,0.03)',
    bordercolor='rgba(0,0,0,0.2)',
    borderwidth=int(SLIDER_BORDERWIDTH),
)

for axis in SENSOR_AXES:
    fig = go.Figure()
    added_classes = []

    for cls in ACTIVE_CLASSES:
        subset = df[df['fan_state'] == cls]
        if subset.empty:
            continue
        y_smooth = _savgol_smooth(
            subset[axis].values,
            sample_rate_hz=EFFECTIVE_SAMPLE_RATE_HZ,
            window_s=SMOOTH_WINDOW_S,
            polyorder=SMOOTH_POLYORDER,
        )
        fig.add_trace(
            go.Scatter(
                x=subset['relative_time_s'],
                y=y_smooth,
                mode='lines',
                name=cls,
                line=temp_line_style(cls),
                hovertemplate=f'<b>{cls}</b><br>Tempo=%{{x:.3f}} s<br>Valor=%{{y:.6f}}<extra></extra>',
            )
        )
        added_classes.append(cls)

    if not added_classes:
        continue

    # --- Botoes de visibilidade (restyle) ---
    all_s   = set(added_classes)
    rot_on  = {c for c in added_classes if c.endswith('ROT_ON')}
    rot_off = {c for c in added_classes if c.endswith('ROT_OFF')}
    baixa   = {c for c in added_classes if c.startswith('LOW_')}
    media   = {c for c in added_classes if c.startswith('MEDIUM_')}
    alta    = {c for c in added_classes if c.startswith('HIGH_')}

    def vis(subset, ac=added_classes):
        return [c in subset for c in ac]

    group_btns = [
        dict(label='Todas',   method='restyle', args=[{'visible': vis(all_s)}]),
        dict(label='ROT ON',  method='restyle', args=[{'visible': vis(rot_on)}]),
        dict(label='ROT OFF', method='restyle', args=[{'visible': vis(rot_off)}]),
        dict(label='Baixa',   method='restyle', args=[{'visible': vis(baixa)}]),
        dict(label='Media',   method='restyle', args=[{'visible': vis(media)}]),
        dict(label='Alta',    method='restyle', args=[{'visible': vis(alta)}]),
    ]

    ind_btns = [
        dict(label=SHORT_TEMPORAL_LABELS.get(c, c), method='restyle',
             args=[{'visible': vis({c})}])
        for c in added_classes
    ]

    # --- Slider de amplitude Y (por tipo de sensor) ---
    if 'gyro' in axis:
        Y_AMP_VALS = [500.0, 250.0, 100.0, 50.0, 20.0]
    else:
        Y_AMP_VALS = [5.0, 2.0, 1.0, 0.5, 0.1]

    y_amp_steps = (
        [dict(label='Auto', method='relayout', args=[{'yaxis.autorange': True}])]
        + [dict(label=f'+-{v:g}', method='relayout',
                args=[{'yaxis.range': [-v, v], 'yaxis.autorange': False}])
           for v in Y_AMP_VALS]
    )

    y_amp_slider = dict(
        active=0,
        currentvalue=dict(prefix='Y amp: ', visible=True, xanchor='left'),
        pad=dict(t=35, b=5, l=10, r=10),
        len=0.55, x=0.22, y=-0.22,
        steps=y_amp_steps,
    )

    fig.update_layout(
        title=dict(
            text=(f'Sinais Sobrepostos (Suavizados) \u2014 {axis}'
                  f'  ({int(EFFECTIVE_SAMPLE_RATE_HZ)} Hz \u00b7 \u2014 ROT ON  \u254c ROT OFF)'),
            x=0.0, xanchor='left', y=_title_y_T, yanchor='bottom', font=dict(size=12),
        ),
        xaxis_title='Tempo (s)',
        yaxis_title=axis,
        hovermode='x unified',
        height=_H_T,
        margin=dict(t=_MT_T, b=_MB_T, r=155, l=60),
        legend=dict(
            x=1.01, y=1.0, xanchor='left', yanchor='top', orientation='v',
            font=dict(size=10), bgcolor='rgba(255,255,255,0.85)',
            bordercolor='rgba(0,0,0,0.15)', borderwidth=1,
        ),
        xaxis=dict(rangeslider=slider_style),
        sliders=[y_amp_slider],
        updatemenus=[
            dict(type='buttons', direction='left', x=0.00, y=1.30,
                 xanchor='left', yanchor='top', showactive=True, pad={'r': 6, 't': 4},
                 buttons=[
                     dict(label='Y linear', method='relayout',
                          args=[{'yaxis.type': 'linear', 'yaxis.autorange': True}]),
                     dict(label='Y log',    method='relayout',
                          args=[{'yaxis.type': 'log',    'yaxis.autorange': True}]),
                 ]),
            dict(type='buttons', direction='left', x=0.00, y=1.56,
                 xanchor='left', yanchor='top', showactive=True, pad={'r': 4, 't': 4},
                 buttons=group_btns),
            dict(type='buttons', direction='left', x=0.00, y=1.82,
                 xanchor='left', yanchor='top', showactive=True, pad={'r': 4, 't': 4},
                 buttons=ind_btns),
        ],
    )
    fig.write_html(os.path.join(OUTPUT_FIGURES, f'03_interactive_sobreposto_{axis}.html'))
    fig.show()
    print(f'  Salvo: 03_interactive_sobreposto_{axis}.html')

In [ ]:
# =============================================================================
# Celula 11: Densidade das features brutas (KDE plots)
# =============================================================================
print('Gerando KDE interativo (Plotly)...')

classes = ACTIVE_CLASSES
colors = ACTIVE_COLOR_MAP

fig = make_subplots(rows=2, cols=3, subplot_titles=SENSOR_AXES)

for i, axis in enumerate(SENSOR_AXES):
    row = i // 3 + 1
    col = i % 3 + 1
    for cls in classes:
        subset = df[df['fan_state'] == cls][axis].dropna()
        if len(subset) < 5:
            continue
        x_min, x_max = subset.min(), subset.max()
        if x_min == x_max:
            x_min -= 1e-6
            x_max += 1e-6
        xs = np.linspace(x_min, x_max, 200)
        try:
            kde = scipy_stats.gaussian_kde(subset)
            ys = kde(xs)
        except Exception:
            continue

        fig.add_trace(
            go.Scatter(
                x=xs, y=ys,
                mode='lines',
                name=cls,
                line=dict(color=colors.get(cls)),
                showlegend=(i == 0),
                hovertemplate=f'Classe={cls}<br>{axis}=%{{x:.6f}}<br>Densidade=%{{y:.6f}}<extra></extra>',
            ),
            row=row, col=col,
        )

    fig.update_xaxes(title_text=axis, row=row, col=col)
    fig.update_yaxes(title_text='Densidade', row=row, col=col)

fig.update_layout(
    title_text='Densidade (KDE) das Features Brutas - Dados Filtrados',
    height=800,
    legend_title_text='Classe',
)
fig.write_html(os.path.join(OUTPUT_FIGURES, '04_densidade_features_brutas.html'))
fig.show()

In [ ]:
# =============================================================================
# Secao B — Assinaturas Espectrais por Classe (FFT por eixo)
#
# Estilo identico ao padrao do projeto:
#   ROT_ON  → linha solida  (cores primarias vibrantes)
#   ROT_OFF → linha tracejada (cores contrastantes sem transparencia)
#   Botoes acima do titulo | sliders de eixo X/Y | legenda vertical direita
# =============================================================================
import importlib
import shared.feature_engineering as _fe_mod2
importlib.reload(_fe_mod2)
from shared.feature_engineering import compute_spectral_signature, SPECTRAL_BANDS

# Parametros da janela espectral (10 s para resolucao Δf = 0.1 Hz)
SPECTRAL_WIN  = min(1000, int(SAMPLE_HZ * 10))
SPECTRAL_STEP = SPECTRAL_WIN // 2
SPECTRAL_NFFT = 4096
N_SPEC_WIN    = 20  # janelas por classe para media do espectro

# Cores: ROT_ON = primarias | ROT_OFF = contrastantes (sem transparencia)
SPECTRAL_COLORS = {
    'FAN_OFF':        '#7f7f7f',
    'LOW_ROT_ON':     '#1f77b4',
    'LOW_ROT_OFF':    '#ff7f0e',
    'MEDIUM_ROT_ON':  '#2ca02c',
    'MEDIUM_ROT_OFF': '#9467bd',
    'HIGH_ROT_ON':    '#d62728',
    'HIGH_ROT_OFF':   '#17becf',
}
SHORT_SPEC_LABELS = {
    'FAN_OFF': 'FAN OFF', 'LOW_ROT_ON': 'LO ON',   'LOW_ROT_OFF':   'LO OFF',
    'MEDIUM_ROT_ON': 'MD ON', 'MEDIUM_ROT_OFF': 'MD OFF',
    'HIGH_ROT_ON': 'HI ON',  'HIGH_ROT_OFF':  'HI OFF',
}
X_MIN_STEPS = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]
Y_MIN_STEPS = [1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]

def spec_line_style(cls):
    is_off = cls.endswith('ROT_OFF')
    return dict(color=SPECTRAL_COLORS.get(cls, '#888'),
                width=2.2 if is_off else 2.0,
                dash='dash' if is_off else 'solid')

def mean_spectrum_cls(df_class, axis):
    n = len(df_class)
    if n < SPECTRAL_WIN:
        return None, None
    starts = np.linspace(0, n - SPECTRAL_WIN,
                         min(N_SPEC_WIN, (n - SPECTRAL_WIN) // SPECTRAL_STEP + 1),
                         dtype=int)
    mags_list = []
    for s in starts:
        vals = df_class[axis].iloc[s:s + SPECTRAL_WIN].values
        freqs, mags, _, _, _ = compute_spectral_signature(
            vals, sampling_hz=SAMPLE_HZ, n_fft=SPECTRAL_NFFT)
        mags_list.append(mags)
    return freqs, np.mean(mags_list, axis=0)

# Parametros de layout (compativel com Plotly antigo — y title em container coords)
_H, _MT, _MB = 720, 310, 185
_plot_bot = _MB / _H
_plot_h   = 1 - _MT / _H - _plot_bot
_title_y  = round(_plot_bot + 1.08 * _plot_h, 4)   # paper 1.08 → container

os.makedirs('output/figures', exist_ok=True)

axes_spec = [a for a in SENSOR_AXES if a in df.columns]

for axis in axes_spec:
    fig = go.Figure()
    peak_table   = []
    added_classes = []

    for cls in ACTIVE_CLASSES:
        df_cls = df[df['fan_state'] == cls].reset_index(drop=True)
        freqs, avg_mags = mean_spectrum_cls(df_cls, axis)
        if freqs is None:
            continue
        added_classes.append(cls)

        mask   = (freqs >= 0.01) & (freqs <= 50.0)
        f_show = freqs[mask];  m_show = avg_mags[mask]

        mask_pk = f_show >= 0.5
        if mask_pk.any():
            idx_pk = int(np.argmax(m_show[mask_pk]))
            peak_f = float(f_show[mask_pk][idx_pk]);  peak_m = float(m_show[mask_pk][idx_pk])
        else:
            peak_f, peak_m = 0.0, 0.0
        peak_table.append({'Classe': cls, 'Pico Hz': round(peak_f, 3), 'Magnitude': round(peak_m, 6)})

        fig.add_trace(go.Scatter(
            x=f_show, y=m_show, name=cls, line=spec_line_style(cls),
            hovertemplate=f'<b>{cls}</b><br>f=%{{x:.3f}} Hz<br>mag=%{{y:.6f}}<extra></extra>',
        ))

    for band, (flo, fhi) in SPECTRAL_BANDS.items():
        if fhi <= 50.0:
            fig.add_vrect(x0=flo, x1=fhi, fillcolor='rgba(180,180,180,0.10)', line_width=0,
                          annotation_text=band, annotation_position='top left', annotation_font_size=10)

    def vis(subset):
        return [c in subset for c in added_classes]

    all_s  = set(added_classes)
    rot_on = {c for c in added_classes if c.endswith('ROT_ON')}
    rot_off= {c for c in added_classes if c.endswith('ROT_OFF')}
    baixa  = {c for c in added_classes if c.startswith('LOW_')}
    media  = {c for c in added_classes if c.startswith('MEDIUM_')}
    alta   = {c for c in added_classes if c.startswith('HIGH_')}

    group_btns = [
        dict(label='Todas',   method='restyle', args=[{'visible': vis(all_s)}]),
        dict(label='ROT ON',  method='restyle', args=[{'visible': vis(rot_on)}]),
        dict(label='ROT OFF', method='restyle', args=[{'visible': vis(rot_off)}]),
        dict(label='Baixa',   method='restyle', args=[{'visible': vis(baixa)}]),
        dict(label='Media',   method='restyle', args=[{'visible': vis(media)}]),
        dict(label='Alta',    method='restyle', args=[{'visible': vis(alta)}]),
    ]
    ind_btns = [
        dict(label=SHORT_SPEC_LABELS.get(c, c), method='restyle', args=[{'visible': vis({c})}])
        for c in added_classes
    ]

    x_slider = dict(active=0,
        currentvalue=dict(prefix='X min: ', suffix=' Hz', visible=True, xanchor='left'),
        pad=dict(t=35, b=5, l=10, r=10), len=0.47, x=0.0, y=-0.26,
        steps=[dict(label=f'{v:.2g} Hz', method='relayout',
                    args=[{'xaxis.type': 'log', 'xaxis.range': [np.log10(v), np.log10(50.0)]}])
               for v in X_MIN_STEPS])
    y_slider = dict(active=1,
        currentvalue=dict(prefix='Y min: ', visible=True, xanchor='left'),
        pad=dict(t=35, b=5, l=10, r=10), len=0.47, x=0.53, y=-0.26,
        steps=[dict(label=f'{v:.0e}', method='relayout',
                    args=[{'yaxis.type': 'log', 'yaxis.range[0]': np.log10(v)}])
               for v in Y_MIN_STEPS])

    fig.update_layout(
        title=dict(
            text=f'Assinaturas Espectrais — {axis}  ({int(SAMPLE_HZ)} Hz · Hann · zero-pad | — ROT ON  ╌ ROT OFF)',
            x=0.0, xanchor='left', y=_title_y, yanchor='bottom', font=dict(size=12),
        ),
        xaxis_title='Frequencia (Hz)', yaxis_title='Magnitude',
        hovermode='x unified', height=_H,
        margin=dict(t=_MT, b=_MB, r=155, l=60),
        legend=dict(x=1.01, y=1.0, xanchor='left', yanchor='top', orientation='v',
                    font=dict(size=10), bgcolor='rgba(255,255,255,0.85)',
                    bordercolor='rgba(0,0,0,0.15)', borderwidth=1),
        sliders=[x_slider, y_slider],
        updatemenus=[
            dict(type='buttons', direction='left', x=0.00, y=1.30, xanchor='left', yanchor='top',
                 showactive=True, pad={'r': 6, 't': 4},
                 buttons=[dict(label='Y linear', method='relayout', args=[{'yaxis.type': 'linear'}]),
                           dict(label='Y log',    method='relayout', args=[{'yaxis.type': 'log'}])]),
            dict(type='buttons', direction='left', x=0.28, y=1.30, xanchor='left', yanchor='top',
                 showactive=True, pad={'r': 6, 't': 4},
                 buttons=[dict(label='X linear', method='relayout', args=[{'xaxis.type': 'linear', 'xaxis.range': [0.01, 50]}]),
                           dict(label='X log',    method='relayout', args=[{'xaxis.type': 'log', 'xaxis.range': [np.log10(0.01), np.log10(50)]}])]),
            dict(type='buttons', direction='left', x=0.00, y=1.56, xanchor='left', yanchor='top',
                 showactive=True, pad={'r': 4, 't': 4}, buttons=group_btns),
            dict(type='buttons', direction='left', x=0.00, y=1.82, xanchor='left', yanchor='top',
                 showactive=True, pad={'r': 4, 't': 4}, buttons=ind_btns),
        ],
    )
    out_f = f'output/figures/01_spectral_{axis}.html'
    fig.write_html(out_f)
    fig.show()

    if peak_table:
        df_peaks = pd.DataFrame(peak_table)
        print(f"\n{'─'*55}")
        print(f"  {axis} — picos dominantes (freqs >= 0.5 Hz):")
        print(df_peaks[['Classe', 'Pico Hz', 'Magnitude']].to_string(index=False))


In [ ]:
# =============================================================================
# Secao C - Exportar dados filtrados para 02_Feature_Engineering
#
# Salva o DataFrame 'df' (apos todos os filtros de qualidade de 01_EDA)
# em dois arquivos:
#   raw_sensor_data_filtered_<ts>.csv   -> arquivo versionado
#   raw_filtered_latest.csv             -> link simbolico para 02_ carregar
# =============================================================================
from datetime import datetime as _dt
import os
from pathlib import Path

_ts_exp = _dt.now().strftime('%Y%m%d_%H%M%S')
_out_dir = Path('output/data')
_out_dir.mkdir(parents=True, exist_ok=True)

_fname_ts = _out_dir / f'raw_sensor_data_filtered_{_ts_exp}.csv'
_fname_latest = _out_dir / 'raw_filtered_latest.csv'

df.to_csv(_fname_ts, index=False)
# Overwrite "latest" pointer
df.to_csv(_fname_latest, index=False)

# Persist the EDA traceability block so 02_ can read it
_eda_meta = {
    'eda_id':          EDA_ID,
    'collection_id':   COLLECTION_ID,
    'eda_timestamp':   _ts_exp,
    'filtered_csv':    str(_fname_ts),
    'rows':            len(df),
    'sample_hz':       float(SAMPLE_HZ),
    'active_classes':  list(ACTIVE_CLASSES),
    # Rastreabilidade da fonte
    'data_source':     DATA_SOURCE if 'DATA_SOURCE' in dir() else 'UNKNOWN',
    'source_table':    (_oracle_table if 'DATA_SOURCE' in dir() and DATA_SOURCE == 'SQL'
                        and '_oracle_table' in dir() else
                        (RAW_SOURCE_PATH if DATA_SOURCE == 'CSV' and 'RAW_SOURCE_PATH' in dir() else None)),
    'export_id':       (EDA_INPUT_MANIFEST.get('export_id') if 'EDA_INPUT_MANIFEST' in dir() else None),
    'raw_source_path': RAW_SOURCE_PATH if 'RAW_SOURCE_PATH' in dir() else None,
}
import json as _json2
_eda_meta_path = _out_dir / 'eda_latest_meta.json'
with open(_eda_meta_path, 'w') as _mf:
    _json2.dump(_eda_meta, _mf, indent=2)

print(f'Exportado: {_fname_ts}  ({len(df):,} linhas)')
print(f'Latest:    {_fname_latest}')
print(f'EDA meta:  {_eda_meta_path}')
print(f'EDA_ID:    {EDA_ID}')


In [ ]:
# =============================================================================
# Celula 12: Engenharia de Features (janela deslizante)
#
# IMPORTANTE: ddof=0 no std (populacional) para alinhar com JavaScript
# 3 metricas por eixo x 2 eixos = 6 features por janela
# =============================================================================

from shared.feature_engineering import extract_features_windowed_basic

# Aplicar sobre os dados FILTRADOS
all_features = []
for cls in ACTIVE_CLASSES:
    df_cls = df[df['fan_state'] == cls].reset_index(drop=True)
    if df_cls.empty:
        print(f'{cls}: 0 janelas (sem dados)')
        continue
    features = extract_features_windowed_basic(
        df_cls,
        cls,
        sensor_axes=FEATURE_AXES,
        window_size=WINDOW_SIZE,
        step_size=STEP_SIZE,
        timestamp_col='timestamp_s',
    )
    all_features.extend(features)
    print(f'{cls}: {len(features)} janelas de {len(df_cls)} amostras')

df_features = pd.DataFrame(all_features)

# Colunas de features (numericas, excluindo metadados)
meta_cols = ['fan_state', 'window_start', 'window_end', 'timestamp_start', 'timestamp_end', 'timestamp_mean']
feature_cols = [c for c in FEATURE_COLUMNS if c in df_features.columns]

print()
print(f'Total de janelas: {len(df_features)}')
print(f'Features por janela: {len(feature_cols)}')
print(f'Distribuicao: {df_features["fan_state"].value_counts().to_dict()}')

In [ ]:
# =============================================================================
# Celula 13: Analise visual das features (boxplot por classe)
# =============================================================================
feature_types = FEATURE_METRICS

class_order = ACTIVE_CLASSES
colors = ACTIVE_COLOR_MAP

for feature_type in feature_types:
    cols_to_plot = [f'{ax}_{feature_type}' for ax in FEATURE_AXES]
    cols_to_plot = [c for c in cols_to_plot if c in feature_cols]
    if not cols_to_plot:
        continue

    df_melted = df_features.melt(
        id_vars=['fan_state'], value_vars=cols_to_plot,
        var_name='feature_axis', value_name='value'
    )
    df_melted['axis'] = df_melted['feature_axis'].str.replace(f'_{feature_type}', '', regex=False)
    df_melted['median'] = df_melted.groupby(['fan_state', 'axis'])['value'].transform('median')

    fig = px.box(
        df_melted,
        x='fan_state', y='value',
        color='fan_state',
        facet_col='axis', facet_col_wrap=3,
        category_orders={'fan_state': class_order},
        color_discrete_map=colors,
        title=f'Boxplot - {feature_type} por eixo',
    )

    fig.update_layout(height=600, width=1200)
    # Eixos independentes (autorange por facet) para evitar boxplots 'achatados'
    fig.update_yaxes(matches=None, autorange=True)
    fig.update_xaxes(tickangle=30)
    fig.write_html(os.path.join(OUTPUT_FIGURES, f'boxplot_{feature_type}.html'))
    try:
        fig.write_image(os.path.join(OUTPUT_FIGURES, f'boxplot_{feature_type}.png'))
    except ValueError as e:
        if 'kaleido' in str(e).lower():
            print('[AVISO] Export PNG ignorado (pacote kaleido nao instalado).')
            print('  Para habilitar: pip install -U kaleido')
        else:
            raise

    fig.show()

In [ ]:
# =============================================================================
# Celula 14: Pair plot das features std (mais discriminativas)
# =============================================================================
pairplot_features = [f'{ax}_std' for ax in FEATURE_AXES]
pairplot_features = [f for f in pairplot_features if f in feature_cols]

if len(pairplot_features) > 1:
    g = sns.pairplot(
        df_features[pairplot_features + ['fan_state']],
        hue='fan_state',
        palette=ACTIVE_COLOR_MAP,
        hue_order=ACTIVE_CLASSES,
        diag_kind='kde',
        plot_kws={'alpha': 0.5, 's': 20},
        corner=True,
    )
    g.figure.suptitle('Pair Plot - Std Features', y=1.02)
    g.figure.savefig(os.path.join(OUTPUT_FIGURES, '05_pairplot_std.png'), dpi=150)
    plt.show()
else:
    print('Poucas features std para pairplot.')

In [ ]:
# =============================================================================
# Celula 14b: Violin plots das features mais discriminativas
# =============================================================================
top_features_for_violin = FEATURE_COLUMNS

class_order = ACTIVE_CLASSES
colors = ACTIVE_COLOR_MAP

df_violin = df_features.melt(
    id_vars=['fan_state'], value_vars=top_features_for_violin,
    var_name='feature', value_name='value'
)
df_violin['median'] = df_violin.groupby(['fan_state', 'feature'])['value'].transform('median')

fig = px.violin(
    df_violin,
    x='fan_state', y='value',
    color='fan_state',
    facet_col='feature', facet_col_wrap=3,
    category_orders={'fan_state': class_order},
    color_discrete_map=colors,
    box=True,
    points='all',
)
fig.update_layout(height=700, width=1200, title='Violin Plots - Features Selecionadas')
# Eixos independentes (autorange por facet) para evitar violins 'achatados'
fig.update_yaxes(matches=None, autorange=True)
fig.update_xaxes(tickangle=30)
fig.write_html(os.path.join(OUTPUT_FIGURES, '06_violin_features.html'))
try:
    fig.write_image(os.path.join(OUTPUT_FIGURES, '06_violin_features.png'))
except ValueError as e:
    if 'kaleido' in str(e).lower():
        print('[AVISO] Export PNG ignorado (pacote kaleido nao instalado).')
        print('  Para habilitar: pip install -U kaleido')
    else:
        raise
fig.show()

In [ ]:
# =============================================================================
# Celula 14c: t-SNE e PCA - Visualizacao de separabilidade das classes
# =============================================================================
from sklearn.preprocessing import StandardScaler

X_viz = df_features[feature_cols].values
y_viz = df_features['fan_state'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_viz)

color_map = ACTIVE_COLOR_MAP

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# PCA 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
for cls in ACTIVE_CLASSES:
    mask = y_viz == cls
    if not mask.any():
        continue
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=color_map.get(cls, '#888'), label=cls, alpha=0.6, s=30)
axes[0].set_title(f'PCA 2D (var explicada: {pca.explained_variance_ratio_.sum()*100:.1f}%)', fontsize=13)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend(fontsize=8)

# t-SNE 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)
for cls in ACTIVE_CLASSES:
    mask = y_viz == cls
    if not mask.any():
        continue
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=color_map.get(cls, '#888'), label=cls, alpha=0.6, s=30)
axes[1].set_title('t-SNE 2D', fontsize=13)
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].legend(fontsize=8)

n_feat = len(feature_cols)
plt.suptitle(f'Separabilidade das Classes ({n_feat} features, {len(ACTIVE_CLASSES)} classes)', fontsize=16, y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_FIGURES, '09_pca_tsne.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Variancia explicada PCA: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, PC2={pca.explained_variance_ratio_[1]*100:.1f}%')

In [ ]:
# =============================================================================
# Celula 15: Matriz de correlacao
# =============================================================================
corr_matrix = df_features[feature_cols].corr()

fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(corr_matrix, cmap='RdBu_r', center=0, ax=ax,
            xticklabels=True, yticklabels=True,
            cbar_kws={'shrink': 0.8})
ax.set_title(f'Matriz de Correlacao ({len(feature_cols)} features)', fontsize=14)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_FIGURES, '07_heatmap_correlacao.png'), dpi=150)
plt.show()

---
**Pipeline de treinamento (ordem de execucao):**

1. `control.html` → coleta de dados → auto-exporta CSV ao finalizar
2. **`01_EDA.ipynb`** → analise temporal (Secoes A) + espectral (Secao B) + exporta `raw_filtered_latest.csv` e `eda_latest_meta.json` (Secao C)
3. **`02_Feature_Engineering.ipynb`** → carrega do `eda_latest_meta.json` (sem reprocessar) → features estendidas + resistentes a deriva + momentos espectrais → exporta modelo
4. **`03_Model_Training_Evaluation.ipynb`** → carrega artefatos do 02 → validacao cruzada → exporta modelo final

**Nota:** O `04_Spectral_Feature_Analysis.ipynb` foi incorporado ao 01_ (Secao B) e ao 02_. Manter como referencia historica apenas.
